# ScFates repeated count reference

This healthy-control candidate reuses notebook 21's two-control transfers, marker exclusions, count-PC15 gene folds, and response-gene scoring. It compares a frozen ScFates backbone with a smooth mean atlas and a conditional negative-binomial count-rate atlas. This is a candidate architecture for review, not an adopted method or evidence about pathology or human tissue.

In [ ]:
from pathlib import Path
import hashlib, io, json, sys
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs
from pseudospace.count_representation import fit_count_representation,transform_count_representation
from pseudospace.repeated_structure import fit_atlas
from pseudospace.trajectory_benchmark import fit_scfates_axis,project_neighbor_axis
from pseudospace.canonical_mean_atlas import fit_mean_atlas,project_mean_atlas
from pseudospace.count_rate_atlas import fit_count_rate_atlas,project_count_rate_atlas
from pseudospace.stage_cache import cached_payload
from importlib.metadata import version
import argparse,os
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root',default=os.environ.get('PSEUDOSPACE_DATA_ROOT',str(REPO/'data')))
parser.add_argument('--results-root',default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'scfates_repeated_count_reference'
OUT.mkdir(parents=True,exist_ok=True)
REFERENCE_OUT=RESULTS_ROOT/'healthy_reconstruction_trajectory_benchmark'
NOTEBOOK_LOGIC_VERSION='22.scfates_count_reference.1'
SEED,N_COMPONENTS,N_FOLDS=15,15,3
CONTROLS=('Ctrl1A2','Ctrl1A4')
GRID=np.linspace(0,1,101)
SCFATES_PARAMS=dict(nodes=30,seed=SEED)
COUNT_ATLAS_PARAMS=dict(grid=GRID,bandwidth=.08,theta=100.,min_effective=12.)
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
VERSIONS={name:version(name) for name in ['scanpy','scFates','anndata','scikit-learn']}
CODE_FILES=[REPO/'pseudospace'/f for f in ['repeated_inputs.py','count_representation.py','trajectory_benchmark.py','repeated_structure.py','canonical_mean_atlas.py','count_rate_atlas.py','stage_cache.py']]
CODE_DIGEST=hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
BASELINE_SCORE_PATH=REFERENCE_OUT/'scores.csv'
if not BASELINE_SCORE_PATH.is_file(): raise FileNotFoundError(f'Required notebook-21 baseline is missing: {BASELINE_SCORE_PATH}')
BASELINE_SCORE_DIGEST=hashlib.sha256(BASELINE_SCORE_PATH.read_bytes()).hexdigest()
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,controls=CONTROLS,seed=SEED,
    components=N_COMPONENTS,folds=N_FOLDS,markers=sorted(MARKERS),grid=GRID.tolist(),
    scfates=SCFATES_PARAMS,mean_atlas=dict(bandwidth=.08,min_effective=12.),
    count_atlas={**COUNT_ATLAS_PARAMS,'grid':GRID.tolist()},
    count_representation_theta=100.,projection_neighbors=15,gene_spline_ridge=.001,
    quarter_read_fraction=.25,quarter_read_seed_rule='seed+fold',
    baseline_score_digest=BASELINE_SCORE_DIGEST,versions=VERSIONS)
print('Installed benchmark versions:',VERSIONS)


## Validated healthy inputs and fixed folds

The input loader and structure-level validation are copied from notebook 21. No query anatomy, coordinates, or physical measurements enter projection.

Individual marker and response-fold genes are excluded from modeling features. Total-library exposure includes all measured genes, so aggregate compositional dependence remains. These tests are not fully independent of the withheld measurements.

In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if set(np.unique(specimens))!=set(CONTROLS): raise ValueError('Expected exactly the two healthy control specimens')
if not np.isin(labels,['S1','S2','S3']).all(): raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[x] for x in labels],dtype=int)
genes=healthy.var_names.to_numpy(str)
X=healthy.layers['lognorm']; X=X.toarray().astype(np.float64,copy=False) if sparse.issparse(X) else np.asarray(X,dtype=np.float64)
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()
if not np.isfinite(X).all() or not np.isfinite(raw_total).all() or np.any(raw_total<=0): raise ValueError('Invalid expression or all-gene exposure')
if 'n_spots' not in obs: raise ValueError('Required n_spots metadata is missing')
n_spots=pd.to_numeric(obs.n_spots,errors='coerce').to_numpy(float)
if not np.isfinite(n_spots).all() or np.any(n_spots<0): raise ValueError('Invalid n_spots metadata')
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(notebook_logic_version=NOTEBOOK_LOGIC_VERSION,config=CACHE_PARAMS,code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),dpt_used_for_fitting=False,depth_used=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab(specimens,labels))
print('Structures:',len(ids),'genes:',len(genes),'outer split: whole specimen')

## Shared held-out-gene evaluation

In [ ]:
def dense_panel(matrix,rows,columns):
    value=matrix[rows][:,columns]
    return value.toarray() if sparse.issparse(value) else np.asarray(value)

def fit_gene_spline(response,z,sample):
    return fit_atlas(response,z,sample,grid=GRID,ridge=.001)

def normalized_variance(response,sample):
    mean=np.mean([response[sample==s].mean(axis=0) for s in np.unique(sample)],axis=0)
    return np.maximum(np.mean([((response[sample==s]-mean)**2).mean(axis=0) for s in np.unique(sample)],axis=0),1e-3)

def primary_rows(response,predictions,variance,query_anatomy,meta):
    out=[]
    for method,pred in predictions.items():
        scaled=(response-pred)**2/variance
        for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
            use=np.ones(len(response),bool) if seg is None else query_anatomy==seg
            if use.any(): out.append(dict(**meta,method=method,segment=name,n=int(use.sum()),normalized_mse=float(scaled[use].mean())))
    return out


## Two-control transfers and paired quarter-read check

Within each transfer and rotating fold, fit the ScFates axis on the source specimen only. Fit both references from that same training axis. Project query expression without labels or refitting. Thin each query structure once at 25% of its all-gene reads, then use that paired thinned profile for all frozen projections. Entropy is conditional model output, not calibrated uncertainty.

All projectors use the same response-gene decoder fitted at the original scFates training positions. Training positions are never reassigned. Thinned predictions are scored against the original response measurements to isolate perturbation of the modeling measurements.

In [ ]:
def run_count_reference():
    score_rows,thin_score_rows,stability_rows,position_rows,diagnostics,failures=[],[],[],[],[],[]
    for source in CONTROLS:
        target=next(s for s in CONTROLS if s!=source)
        tr,te=specimens==source,specimens==target
        for fold in range(N_FOLDS):
            key=dict(source_specimen=source,test_specimen=target,fold=fold)
            mg=np.flatnonzero(~marker_mask & (gene_fold!=fold))
            eg=np.flatnonzero(~marker_mask & (gene_fold==fold))
            try:
                train_counts=dense_panel(raw_counts,tr,mg)
                query_counts=dense_panel(raw_counts,te,mg)
                Y,transform=fit_count_representation(train_counts,raw_total[tr],specimens[tr],
                    N_COMPONENTS,theta=100.,seed=SEED)
                query_Y=transform_count_representation(query_counts,raw_total[te],transform)
                train_response=X[tr][:,eg]
                query_response=X[te][:,eg]
                variance=normalized_variance(train_response,specimens[tr])

                train_z,scf_meta=fit_scfates_axis(Y,anatomy[tr],**SCFATES_PARAMS)
                scf_query_z=project_neighbor_axis(Y,train_z,query_Y,k=15)
                decoder=fit_gene_spline(train_response,train_z,specimens[tr])
                predictions={'scFates_neighbor':decoder.predict(scf_query_z)}
                query_coordinates={'scFates_neighbor':scf_query_z}
                mean_atlas=fit_mean_atlas(Y,train_z,grid=GRID,bandwidth=.08,min_effective=12)
                mean_query=project_mean_atlas(mean_atlas,query_Y)
                predictions['mean_reference']=decoder.predict(mean_query['z_mean'])
                query_coordinates['mean_reference']=mean_query['z_mean']
                rate_atlas=fit_count_rate_atlas(train_counts,raw_total[tr],train_z,specimens[tr],**COUNT_ATLAS_PARAMS)
                rate_query=project_count_rate_atlas(rate_atlas,query_counts,raw_total[te])
                predictions['count_NB_reference']=decoder.predict(rate_query['z_mean'])
                query_coordinates['count_NB_reference']=rate_query['z_mean']
                score_rows.extend(primary_rows(query_response,predictions,variance,anatomy[te],key))

                # Thin each query structure once using the full measured count matrix;
                # the same paired realization feeds all three frozen projectors.
                query_full=sparse.csr_matrix(raw_counts[te])
                rng=np.random.default_rng(SEED+fold)
                thin_full=sparse.csr_matrix((rng.binomial(query_full.data.astype(np.int64),.25),
                    query_full.indices.copy(),query_full.indptr.copy()),shape=query_full.shape)
                thin_library=np.asarray(thin_full.sum(axis=1)).ravel().astype(float)
                thin_model=dense_panel(thin_full,np.ones(thin_full.shape[0],dtype=bool),mg)
                thin_Y=transform_count_representation(thin_model,thin_library,transform)
                thin_scf_z=project_neighbor_axis(Y,train_z,thin_Y,k=15)
                thin_mean=project_mean_atlas(mean_atlas,thin_Y)
                thin_rate=project_count_rate_atlas(rate_atlas,thin_model,thin_library)
                thin_coordinates={'scFates_neighbor':thin_scf_z,
                    'mean_reference':thin_mean['z_mean'],
                    'count_NB_reference':thin_rate['z_mean']}
                thin_predictions={'scFates_neighbor':decoder.predict(thin_scf_z),
                    'mean_reference':decoder.predict(thin_mean['z_mean']),
                    'count_NB_reference':decoder.predict(thin_rate['z_mean'])}
                thin_score_rows.extend(primary_rows(query_response,thin_predictions,variance,anatomy[te],key))
                for method,zq in query_coordinates.items():
                    zthin=thin_coordinates[method]
                    movement=np.abs(zthin-zq)
                    from scipy.stats import spearmanr
                    for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
                        use=np.ones(len(zq),bool) if seg is None else anatomy[te]==seg
                        rho=spearmanr(zq[use],zthin[use]).statistic if use.sum()>1 else np.nan
                        row=dict(**key,method=method,segment=name,n=int(use.sum()),
                            coordinate_mae=float(movement[use].mean()),fraction_moved_over_0_1=float(np.mean(movement[use]>.1)),
                            within_segment_spearman=float(rho) if np.isfinite(rho) else np.nan)
                        if method=='mean_reference': row['conditional_entropy_mean']=float(mean_query['entropy'][use].mean())
                        elif method=='count_NB_reference': row['conditional_entropy_mean']=float(rate_query['entropy'][use].mean())
                        else: row['conditional_entropy_mean']=np.nan
                        row['thinned_conditional_entropy_mean']=(float(thin_mean['entropy'][use].mean()) if method=='mean_reference' else float(thin_rate['entropy'][use].mean()) if method=='count_NB_reference' else np.nan)
                        stability_rows.append(row)
                    entropy=mean_query['entropy'] if method=='mean_reference' else (rate_query['entropy'] if method=='count_NB_reference' else np.full(len(zq),np.nan))
                    for structure_id,zvalue,hvalue,disagreement in zip(ids[te],zq,entropy,np.abs(zq-scf_query_z)):
                        position_rows.append(dict(**key,method=method,structure_id=structure_id,z=float(zvalue),conditional_entropy=float(hvalue) if np.isfinite(hvalue) else np.nan,coordinate_disagreement_from_scfates=float(disagreement)))
                diagnostics.append(dict(**key,status='complete',scfates_metadata=json.dumps(scf_meta),
                    training_n=len(train_z),query_n=len(scf_query_z),model_genes=len(mg),response_genes=len(eg),
                    mean_reference_sigma2=float(mean_atlas['sigma2']),count_theta=float(rate_atlas['theta'])))
                print('Completed source→target:',source,'→',target,'fold',fold,flush=True)
            except Exception as exc:
                failures.append(dict(**key,error=str(exc)))
                diagnostics.append(dict(**key,status='failed',scfates_metadata='',training_n=np.nan,
                    query_n=np.nan,model_genes=len(mg),response_genes=len(eg),mean_reference_sigma2=np.nan,count_theta=np.nan))
                print('Failed source→target:',source,'→',target,'fold',fold,str(exc),flush=True)
    frames=dict(scores=score_rows,thin_scores=thin_score_rows,stability=stability_rows,
        positions=position_rows,diagnostics=diagnostics,failures=failures)
    return {name:pd.DataFrame(rows).to_json(orient='table') for name,rows in frames.items()}


In [ ]:
CACHE_INPUTS=dict(X=X,counts=raw_counts,library=raw_total,ids=ids,specimens=specimens,
    labels=labels,n_spots=n_spots,gene_fold=gene_fold,markers=marker_mask,
    baseline_score_digest=BASELINE_SCORE_DIGEST)
payload=cached_payload('scfates_repeated_count_reference',run_count_reference,
    root=OUT/'stage_cache',params=CACHE_PARAMS,inputs=CACHE_INPUTS,
    code=CODE_DIGEST)
outputs={name:pd.read_json(io.StringIO(str(value)),orient='table') for name,value in payload.items()}
for name,frame in outputs.items(): frame.to_csv(OUT/(name+'.csv'),index=False)
# Re-run completeness and baseline-consistency guards on cache hits as well.
if len(outputs['diagnostics'])!=len(CONTROLS)*N_FOLDS or len(outputs['failures']):
    raise RuntimeError(f"Expected six complete references; failures={len(outputs['failures'])}")
reference_scores=pd.read_csv(BASELINE_SCORE_PATH)
current=outputs['scores'].query("method=='scFates_neighbor'").sort_values(['source_specimen','test_specimen','fold','segment']).reset_index(drop=True)
expected=reference_scores.query("method=='scFates_neighbor'").sort_values(['source_specimen','test_specimen','fold','segment']).reset_index(drop=True)
keys=['source_specimen','test_specimen','fold','segment','n']
if len(current)!=len(expected) or not current[keys].equals(expected[keys]) or not np.allclose(current.normalized_mse,expected.normalized_mse,rtol=1e-8,atol=1e-10):
    raise RuntimeError('scFates_neighbor did not reproduce notebook-21 scores within rtol=1e-8')
display(outputs['failures'])
display(outputs['diagnostics'])
display(outputs['scores'].pivot_table(index=['source_specimen','fold','segment'],columns='method',values='normalized_mse'))
display(outputs['stability'].pivot_table(index=['source_specimen','fold','method','segment'],values=['coordinate_mae','fraction_moved_over_0_1','within_segment_spearman','conditional_entropy_mean']))
print('All six training-only references completed; notebook-21 scFates scores reproduced.')

# Gene folds are overlapping feature perturbations, not independent replications.
panel=outputs['positions'].groupby(['source_specimen','test_specimen','method','structure_id']).agg(
    n_folds=('fold','nunique'),z_min=('z','min'),z_max=('z','max'),
    mean_conditional_entropy=('conditional_entropy','mean')).reset_index()
if not panel.n_folds.eq(N_FOLDS).all(): raise RuntimeError('Incomplete gene-panel sensitivity')
panel['z_range']=panel.z_max-panel.z_min
panel.to_csv(OUT/'gene_panel_sensitivity.csv',index=False)
display(panel.groupby(['source_specimen','method']).z_range.agg(['mean','median','max']))

from scipy.stats import spearmanr
panel_rank_rows=[]
segment_lookup=dict(zip(ids,np.asarray(['S1','S2','S3'])[anatomy]))
for (source,method),rows in outputs['positions'].groupby(['source_specimen','method']):
    pairs=rows.pivot(index='structure_id',columns='fold',values='z')
    segment=np.asarray([segment_lookup[x] for x in pairs.index])
    for name in ['all','S1','S2','S3']:
        use=np.ones(len(pairs),bool) if name=='all' else segment==name
        for first,second in [(0,1),(0,2),(1,2)]:
            panel_rank_rows.append(dict(source_specimen=source,method=method,segment=name,
                first_fold=first,second_fold=second,n=int(use.sum()),
                spearman=float(spearmanr(pairs.loc[use,first],pairs.loc[use,second]).statistic),
                coordinate_mae=float(np.mean(np.abs(pairs.loc[use,first]-pairs.loc[use,second])))))
panel_ranks=pd.DataFrame(panel_rank_rows)
panel_ranks.to_csv(OUT/'gene_panel_pairwise_ranks.csv',index=False)
display(panel_ranks.groupby(['source_specimen','method','segment']).spearman.agg(['min','max']))


## Prediction and coordinate sensitivity

In [ ]:
import matplotlib.pyplot as plt
score=outputs['scores'].query("segment=='all'")
base=score.query("method=='scFates_neighbor'")[['source_specimen','fold','normalized_mse']].rename(columns={'normalized_mse':'baseline_mse'})
comparison=score.merge(base,on=['source_specimen','fold'])
comparison['gain_percent']=100*(1-comparison.normalized_mse/comparison.baseline_mse)
comparison.to_csv(OUT/'relative_gain_vs_scfates.csv',index=False)
models=['scFates_neighbor','mean_reference','count_NB_reference']
colors={'scFates_neighbor':'C0','mean_reference':'C1','count_NB_reference':'C2'}
fig,axes=plt.subplots(2,2,figsize=(11,7))
for column,source in enumerate(CONTROLS):
    for i,method in enumerate(models):
        vals=comparison.query('source_specimen==@source and method==@method').gain_percent.to_numpy()
        axes[0,column].scatter(np.full(len(vals),i),vals,color=colors[method],s=32)
        axes[0,column].plot([i-.15,i+.15],[vals.mean()]*2,color=colors[method])
        moves=outputs['stability'].query('source_specimen==@source and method==@method and segment=="all"').coordinate_mae.to_numpy()
        axes[1,column].scatter(np.full(len(moves),i),moves,color=colors[method],s=32)
        axes[1,column].plot([i-.15,i+.15],[moves.mean()]*2,color=colors[method])
    axes[0,column].axhline(0,color='gray',linewidth=.7)
    axes[0,column].set_title(source+' → held-out control')
    for ax in axes[:,column]: ax.set_xticks(range(len(models)),['scFates + neighbors','mean reference','NB count reference'],rotation=15,ha='right',fontsize=10)
axes[0,0].set_ylabel('Gene error reduction vs scFates NN (%)',fontsize=10)
axes[1,0].set_ylabel('Mean coordinate change at quarter reads',fontsize=10)
fig.suptitle('Fixed backbone and decoder; dots are gene folds, not mice')
fig.tight_layout();fig.savefig(OUT/'reference_prediction_and_thinning.png',dpi=160);plt.show()


## Interpretation ceiling

These are exploratory held-out prediction and read-thinning diagnostics on the same two healthy controls used during method development. The frozen count atlas assumes independent negative-binomial gene counts conditional on total library exposure and a fixed dispersion. Neither its posterior entropy nor a sharper posterior establishes anatomical precision. No adoption decision or pathology/human conclusion is made here.

The actual-data run found worse all-gene prediction for both alternative projectors in all folds. NB coordinate movement under quarter-read thinning is smaller, but that does not validate location. Gene-panel shifts include numerical stretching and changed within-segment order; S1 ordering is particularly sensitive in the Ctrl1A4-trained reference. Coarse global rank and sharp conditional posteriors do not resolve fine-position ambiguity. These are single-source validation experiments inside the repeated-object framework, not a final joint atlas fitted from all specimens.


## Frozen training-selected positional programs

Reuse notebook 18’s source-only cross-fit gene selection panels without reading its historical query-error columns. Re-score those identical panels with the fixed ScFates-neighbor, mean-reference, and count-NB coordinates, and compare them with a degree-2 segment-plus-coverage oracle. Panels favor an earlier atlas-development procedure; query anatomy is used only by the explicitly privileged oracle and scoring strata. These comparisons do not establish anatomical truth or select an adopted method.

In [ ]:
# Independent stage: consumes only frozen training-selected panel identifiers/gains.
from pseudospace.within_segment_validation import fit_segment_covariate_oracle, predict_segment_covariate_oracle
from scipy.stats import spearmanr

PANEL_PATH=RESULTS_ROOT/'within_segment_repeated_structure_validation'/'selected_genes.csv'
if not PANEL_PATH.is_file(): raise FileNotFoundError(f'Required notebook-18 selected panel is missing: {PANEL_PATH}')
PANEL_COLUMNS=['source_specimen','test_specimen','fold','segment','gene','training_cv_gain']
frozen_panel=pd.read_csv(PANEL_PATH,usecols=PANEL_COLUMNS)
if frozen_panel.empty: raise ValueError('Notebook-18 frozen selected-gene panel is empty')
expected_pairs={(s,next(t for t in CONTROLS if t!=s)) for s in CONTROLS}
if set(zip(frozen_panel.source_specimen,frozen_panel.test_specimen))!=expected_pairs:
    raise ValueError('Frozen panels do not contain the expected two source→test transfers')
if set(frozen_panel.fold.astype(int))!={0,1,2} or not set(frozen_panel.segment).issubset({'S1','S2','S3'}):
    raise ValueError('Frozen panel fold or segment labels are invalid')
if frozen_panel.duplicated(['source_specimen','test_specimen','fold','segment','gene']).any():
    raise ValueError('Frozen panel contains duplicate gene rows within a segment panel')
if not np.isfinite(frozen_panel.training_cv_gain.to_numpy(float)).all():
    raise ValueError('Frozen training CV gains must be finite')

FIXED_PROGRAM_VERSION='22.fixed_program.1'
PANEL_CODE_DIGEST=hashlib.sha256(CODE_DIGEST.encode()+
    (REPO/'pseudospace'/'within_segment_validation.py').read_bytes()).hexdigest()
PANEL_PARAMS=dict(version=FIXED_PROGRAM_VERSION,base_version=NOTEBOOK_LOGIC_VERSION,
    selected_panel_source='notebook18_training_cv_selected_genes_only',folds=N_FOLDS,
    selected_per_segment=50,coordinate_source='outputs.positions_expression_only',
    oracle=dict(degree=2,ridge=.01,covariates=['log_allgene_library','log1p_n_spots','log1p_detected_model_genes']),
    variance_floor=1e-3,scfates_reconstruction_rtol=1e-8)

def run_fixed_program_evaluation():
    score_rows,gene_rows,diagnostics,failures=[],[],[],[]
    for source in CONTROLS:
        target=next(t for t in CONTROLS if t!=source)
        tr,te=specimens==source,specimens==target
        for fold in range(N_FOLDS):
            key=dict(source_specimen=source,test_specimen=target,fold=fold)
            try:
                fold_panel=frozen_panel[(frozen_panel.source_specimen==source)&
                    (frozen_panel.test_specimen==target)&(frozen_panel.fold==fold)].copy()
                if fold_panel.empty: raise ValueError('Missing frozen source/fold panel')
                mg=np.flatnonzero(~marker_mask&(gene_fold!=fold))
                eg=np.flatnonzero(~marker_mask&(gene_fold==fold))
                response_gene_names=genes[eg]
                gene_to_j={g:j for j,g in enumerate(response_gene_names)}
                for segname in ['S1','S2','S3']:
                    panel_seg=fold_panel[fold_panel.segment==segname]
                    if panel_seg.empty: raise ValueError(f'Frozen {segname} panel is empty')
                    if panel_seg.gene.duplicated().any(): raise ValueError(f'Duplicate gene in frozen {segname} panel')
                    if not set(panel_seg.gene).issubset(gene_to_j):
                        raise ValueError(f'{segname} panel contains genes outside this held-out response fold')

                train_counts=dense_panel(raw_counts,tr,mg)
                query_counts=dense_panel(raw_counts,te,mg)
                Y,transform=fit_count_representation(train_counts,raw_total[tr],specimens[tr],
                    N_COMPONENTS,theta=100.,seed=SEED)
                query_Y=transform_count_representation(query_counts,raw_total[te],transform)
                train_z,scf_meta=fit_scfates_axis(Y,anatomy[tr],**SCFATES_PARAMS)
                reconstructed_scf=project_neighbor_axis(Y,train_z,query_Y,k=15)

                # Align cached expression-only coordinates by structure ID and check ScFates reconstruction.
                saved=outputs['positions'][(outputs['positions'].source_specimen==source)&
                    (outputs['positions'].test_specimen==target)&(outputs['positions'].fold==fold)]
                methods=['scFates_neighbor','mean_reference','count_NB_reference']
                coordinates={}
                for method in methods:
                    rows=saved[saved.method==method]
                    if rows.structure_id.duplicated().any() or set(rows.structure_id)!=set(ids[te]):
                        raise ValueError(f'{method} saved positions do not align exactly with query structure IDs')
                    lookup=dict(zip(rows.structure_id,rows.z.astype(float)))
                    coordinates[method]=np.asarray([lookup[x] for x in ids[te]],dtype=float)
                max_coordinate_error=float(np.max(np.abs(coordinates['scFates_neighbor']-reconstructed_scf)))
                if not np.allclose(coordinates['scFates_neighbor'],reconstructed_scf,rtol=1e-8,atol=1e-10):
                    raise ValueError(f'ScFates-neighbor coordinate reconstruction failed (max abs error {max_coordinate_error:g})')

                train_response=X[tr][:,eg]
                query_response=X[te][:,eg]
                train_sample=specimens[tr]
                decoder=fit_gene_spline(train_response,train_z,train_sample)
                predictions={method:decoder.predict(coordinates[method]) for method in methods}
                variance=normalized_variance(train_response,train_sample)
                # Reproduce notebook-22's all-response-fold ScFates scores before using selected panels.
                reproduced=primary_rows(query_response,{'scFates_neighbor':predictions['scFates_neighbor']},
                    variance,anatomy[te],key)
                reproduced=pd.DataFrame(reproduced)
                expected=outputs['scores'][(outputs['scores'].source_specimen==source)&
                    (outputs['scores'].test_specimen==target)&(outputs['scores'].fold==fold)&
                    (outputs['scores'].method=='scFates_neighbor')]
                checked=reproduced.merge(expected[['segment','n','normalized_mse']],on='segment',
                    suffixes=('_new','_cached'),validate='one_to_one')
                if len(checked)!=4 or not checked.n_new.equals(checked.n_cached) or not np.allclose(
                        checked.normalized_mse_new,checked.normalized_mse_cached,rtol=1e-8,atol=1e-10):
                    raise ValueError('All-response-fold ScFates normalized MSE did not reproduce notebook-22 scores')

                train_detected=np.asarray((raw_counts[tr][:,mg]>0).sum(axis=1)).ravel()
                query_detected=np.asarray((raw_counts[te][:,mg]>0).sum(axis=1)).ravel()
                train_cov=np.column_stack([np.log(raw_total[tr]),np.log1p(n_spots[tr]),np.log1p(train_detected)])
                query_cov=np.column_stack([np.log(raw_total[te]),np.log1p(n_spots[te]),np.log1p(query_detected)])
                oracle=fit_segment_covariate_oracle(train_response,train_cov,anatomy[tr],train_sample,
                    ridge=.01,degree=2)
                oracle_prediction=predict_segment_covariate_oracle(oracle,query_cov,anatomy[te])
                predictions['quadratic_segment_coverage_oracle']=oracle_prediction

                for seg,segname in enumerate(['S1','S2','S3']):
                    selected=fold_panel[fold_panel.segment==segname]
                    jj=np.asarray([gene_to_j[g] for g in selected.gene],dtype=int)
                    train_segment=anatomy[tr]==seg
                    query_segment=anatomy[te]==seg
                    if not train_segment.any() or not query_segment.any():
                        raise ValueError(f'No source or query profiles for {segname} scoring')
                    scale=np.maximum(np.var(train_response[train_segment],axis=0),1e-3)
                    for method,prediction in predictions.items():
                        normalized=((query_response[query_segment][:,jj]-prediction[query_segment][:,jj])**2)/scale[jj]
                        score_rows.append(dict(**key,method=method,segment=segname+'_selected',
                            n=int(query_segment.sum()),selected_gene_count=int(len(jj)),normalized_mse=float(normalized.mean())))
                        for gene_idx,gene in zip(jj,selected.gene):
                            gene_loss=float(np.mean((query_response[query_segment,gene_idx]-prediction[query_segment,gene_idx])**2)/scale[gene_idx])
                            gene_rows.append(dict(**key,method=method,segment=segname,gene=gene,
                                training_cv_gain=float(selected.loc[selected.gene==gene,'training_cv_gain'].iloc[0]),
                                normalized_mse=gene_loss))
                diagnostics.append(dict(**key,status='complete',panel_sizes={s:int((fold_panel.segment==s).sum()) for s in ['S1','S2','S3']},
                    max_scfates_coordinate_abs_error=max_coordinate_error,
                    max_scfates_all_gene_score_abs_error=float(np.max(np.abs(checked.normalized_mse_new-checked.normalized_mse_cached))),
                    decoder_axis='original training scFates z',oracle_degree=2,scfates_metadata=json.dumps(scf_meta)))
                print('Completed frozen-panel transfer:',source,'→',target,'fold',fold,flush=True)
            except Exception as exc:
                failures.append(dict(**key,error=str(exc)))
                diagnostics.append(dict(**key,status='failed',panel_sizes='',max_scfates_coordinate_abs_error=np.nan,
                    max_scfates_all_gene_score_abs_error=np.nan,decoder_axis='original training scFates z',oracle_degree=2,scfates_metadata=''))
                print('Failed frozen-panel transfer:',source,'→',target,'fold',fold,str(exc),flush=True)
    frames={'scores':pd.DataFrame(score_rows),'gene_errors':pd.DataFrame(gene_rows),
        'diagnostics':pd.DataFrame(diagnostics),'failures':pd.DataFrame(failures)}
    return {name:frame.to_json(orient='table') for name,frame in frames.items()}

PANEL_CACHE_INPUTS={**CACHE_INPUTS,'filtered_frozen_panel':frozen_panel,
    'saved_expression_only_positions':outputs['positions']}
panel_payload=cached_payload('scfates_repeated_count_reference_fixed_program',
    run_fixed_program_evaluation,root=OUT/'stage_cache',params=PANEL_PARAMS,
    inputs=PANEL_CACHE_INPUTS,code=PANEL_CODE_DIGEST)
panel_outputs={name:pd.read_json(io.StringIO(str(value)),orient='table')
    for name,value in panel_payload.items()}
for name,frame in panel_outputs.items():frame.to_csv(OUT/('fixed_program_'+name+'.csv'),index=False)
if len(panel_outputs['diagnostics'])!=len(CONTROLS)*N_FOLDS or len(panel_outputs['failures']):
    raise RuntimeError(f"Expected six complete frozen-panel fits; failures={len(panel_outputs['failures'])}")
display(panel_outputs['failures'])
display(panel_outputs['diagnostics'])
wide=panel_outputs['scores'].pivot_table(index=['source_specimen','test_specimen','fold','segment'],
    columns='method',values='normalized_mse')
comparison=panel_outputs['scores'].copy()
comparison['quadratic_mse']=[wide.loc[(r.source_specimen,r.test_specimen,r.fold,r.segment),'quadratic_segment_coverage_oracle'] for r in comparison.itertuples()]
comparison['scfates_mse']=[wide.loc[(r.source_specimen,r.test_specimen,r.fold,r.segment),'scFates_neighbor'] for r in comparison.itertuples()]
comparison['gain_vs_quadratic_pct']=100*(comparison.quadratic_mse-comparison.normalized_mse)/comparison.quadratic_mse
comparison['gain_vs_scfatesNN_pct']=100*(comparison.scfates_mse-comparison.normalized_mse)/comparison.scfates_mse
comparison.to_csv(OUT/'fixed_program_comparison.csv',index=False)
display(comparison.groupby(['source_specimen','segment','method'])[['normalized_mse','gain_vs_quadratic_pct','gain_vs_scfatesNN_pct']].agg(['mean','median','count']))
print('Folds are descriptive transfer repeats; panels are frozen training selections, not independent validations.')